In [3]:
%run stage1_multimodal.py \
    --data_dir /home/maweicheng/ST_Graduation_Project/database \
    --output_dir ./stage1_multimodal_results \
    --phase1_epochs 100 \
    --phase2_epochs 100 \
    --phase3_epochs 50 \
    --batch_size 256 \
    --lr 1e-3 \
    --latent_dim 128 \
    --fusion_method concat \
    --align_method mmd \
    --beta 1.0

Using device: cuda
Loading datasets...
   Found samples: ['CID4290', 'CID4465', 'CID44971', 'CID4535']
   ✓ Loaded CID4290: SC (5082, 27719), ST (2427, 23116)
   ✓ Loaded CID4465: SC (1561, 27719), ST (1203, 23157)
   ✓ Loaded CID4290: SC (5082, 27719), ST (2427, 23116)
   ✓ Loaded CID4465: SC (1561, 27719), ST (1203, 23157)
   ✓ Loaded CID44971: SC (7740, 27719), ST (1147, 23531)
   ✓ Loaded CID4535: SC (3641, 27719), ST (1122, 22331)
   ✓ Loaded CID44971: SC (7740, 27719), ST (1147, 23531)
   ✓ Loaded CID4535: SC (3641, 27719), ST (1122, 22331)
   SC total: (18024, 27719)
   ST total: (5899, 19632)

Computing clusters and marker genes...
Starting clustering analysis...
   SC total: (18024, 27719)
   ST total: (5899, 19632)

Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 25 clusters
Clustering results: 25 clusters
Marker genes per cluster:
Total: 1334 marker genes
Total marker genes: 1334
Extracting image patches (size=112)...
Returning dumm

KeyboardInterrupt: 

## 6. 加载训练好的模型

In [ ]:
# 加载对齐后的模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = "./stage1_multimodal_results"

# 假设你知道基因数量（从 marker_genes.txt）
marker_genes_file = os.path.join(output_dir, "marker_genes.txt")
with open(marker_genes_file, 'r') as f:
    marker_genes = [line.strip() for line in f.readlines()]
n_genes = len(marker_genes)
latent_dim = 128

print(f"Number of marker genes: {n_genes}")

# 加载 SC VAE
sc_vae = VAE(input_dim=n_genes, latent_dim=latent_dim).to(device)
sc_vae.load_state_dict(torch.load(os.path.join(output_dir, 'sc_vae_aligned.pth')))
sc_vae.eval()
print("✓ SC VAE loaded")

# 加载 ST Multi-Modal VAE
st_vae = MultiModalVAE(gene_input_dim=n_genes, latent_dim=latent_dim).to(device)
st_vae.load_state_dict(torch.load(os.path.join(output_dir, 'st_multimodal_vae_aligned.pth')))
st_vae.eval()
print("✓ ST Multi-Modal VAE loaded")

## 7. 提取 Latent 表示并可视化

In [ ]:
# 准备数据
sc_adata = sc.read_h5ad("path/to/sc_data.h5ad")
st_adata = sc.read_h5ad("path/to/st_data_with_image.h5ad")

# 提取 SC latent
sc_adata_count = sc_adata.copy()
sc.pp.normalize_total(sc_adata_count, target_sum=1e4)
sc_subset = sc_adata_count[:, marker_genes].copy()
sc_X = sc_subset.X.toarray() if hasattr(sc_subset.X, 'toarray') else sc_subset.X

with torch.no_grad():
    sc_X_tensor = torch.FloatTensor(sc_X).to(device)
    sc_latent, _, _ = sc_vae.encode(sc_X_tensor)
    sc_latent = sc_latent.cpu().numpy()

print(f"SC latent shape: {sc_latent.shape}")

# 提取 ST latent（需要图像数据）
# ... (类似的处理)

# 使用 UMAP 可视化
import umap

# 合并 SC 和 ST latent
# combined_latent = np.vstack([sc_latent, st_latent])
# labels = ['SC'] * len(sc_latent) + ['ST'] * len(st_latent)

# reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
# embedding = reducer.fit_transform(combined_latent)

# # 可视化
# plt.figure(figsize=(10, 8))
# for label in ['SC', 'ST']:
#     mask = np.array(labels) == label
#     plt.scatter(embedding[mask, 0], embedding[mask, 1], label=label, alpha=0.6)
# plt.legend()
# plt.title('SC-ST Latent Space (UMAP)')
# plt.xlabel('UMAP 1')
# plt.ylabel('UMAP 2')
# plt.show()

## 8. 不同融合方法对比

In [ ]:
# 测试不同的融合方法
fusion_methods = ['concat', 'add', 'product']

for method in fusion_methods:
    print(f"\nTraining with fusion method: {method}")
    !conda run -n dl python stage1_multimodal.py \
        --output_dir ./stage1_multimodal_results_{method} \
        --phase1_epochs 30 \
        --phase2_epochs 30 \
        --phase3_epochs 15 \
        --fusion_method {method} \
        --batch_size 128

## 9. 应用到 Stage 2

训练完成后，可以在 Stage 2 中使用对齐后的模型进行解卷积。

In [ ]:
# Stage 2 会自动加载 Stage 1 的模型
# 确保在 Stage 2 中指向正确的 Stage 1 输出目录

# !conda run -n dl python stage2.py \
#     --stage1_dir ./stage1_multimodal_results \
#     --st_file path/to/st_data_with_image.h5ad \
#     --output_dir ./stage2_results \
#     --n_epochs 200